# Enhanced BTC Consumer
- Filters incoming data
- Periodically saves data in batches
- Includes robust error handling and logging
- Live visualization of BTC price over time

In [ ]:
import json
import time
import pandas as pd
import matplotlib.pyplot as plt
from pykafka import KafkaClient
from pykafka.exceptions import NoBrokersAvailableError
import os
import logging

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Settings
BROKER = 'localhost:9092'
TOPIC = 'btc_topic'
BATCH_SIZE = 10
CSV_FILE = 'btc_data.csv'

# Live plot setup
%matplotlib notebook
plt.ion()
fig, ax = plt.subplots()
x_data, y_data = [], []
line, = ax.plot(x_data, y_data, 'b-')
ax.set_xlabel('Timestamp')
ax.set_ylabel('BTC Price')
ax.set_title('Live BTC Price Stream')

def update_plot(ts, price):
    x_data.append(pd.to_datetime(ts, unit='s'))
    y_data.append(price)
    line.set_data(x_data, y_data)
    ax.relim()
    ax.autoscale_view()
    fig.canvas.draw()
    fig.canvas.flush_events()

def run_consumer():
    try:
        client = KafkaClient(hosts=BROKER)
        topic = client.topics[TOPIC.encode()]
        consumer = topic.get_simple_consumer(reset_offset_on_start=True)

        logging.info('Consumer started')
        data_batch = []

        for message in consumer:
            if message is None:
                continue
            try:
                data = json.loads(message.value.decode())
                price = data.get('price')
                timestamp = data.get('timestamp')

                # Filter out unreasonable prices
                if price is not None and 10000 < price < 200000:
                    update_plot(timestamp, price)
                    data_batch.append({'timestamp': timestamp, 'price': price})

                    if len(data_batch) >= BATCH_SIZE:
                        df = pd.DataFrame(data_batch)
                        if os.path.exists(CSV_FILE):
                            df.to_csv(CSV_FILE, mode='a', header=False, index=False)
                        else:
                            df.to_csv(CSV_FILE, index=False)
                        logging.info(f'Saved {len(data_batch)} records to {CSV_FILE}')
                        data_batch = []
            except Exception as e:
                logging.error(f'Error processing message: {e}')
    except NoBrokersAvailableError:
        logging.critical('Kafka broker not available. Please check your Kafka setup.')

# Run the consumer
run_consumer()